# FOXF1_bead — 08_fixed_foxf1_dip_review

**Feeds:** Fig 2c, ED Fig 3d

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 08 Fixed FOXF1 Dip Review

## Strategy

This notebook focuses on the fixed `FOXF1-YFP / DAPI` dip around `~750 um` in the nearest-bead trace. The goal is not to recompute the full fixed pipeline, but to use the current approved `07` outputs to identify which spatial regions in `1-2` and `3-2` are pulling those distance bins downward.

Approach:
- use the current final `04` masks and saved `07` fixed background parameters
- rebuild the per-pixel fixed `FOXF1 bgz / DAPI` image for the dip-driving positions
- highlight the exact `700-800 um` bead-distance window
- compute a pixelwise residual image: `pixel ratio - cohort mean for that exact distance bin`
- show the negative-residual pixels that directly pull the trace down

The visual style follows the masked ratio images in `07`, since those are the most interpretable for this question.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

MPLCONFIGDIR = ROOT / ".matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIGDIR)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print("ROOT:", ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from scripts import pipeline_common as common
from scripts import live_fixed_quantification as lfq


## Paths And Configuration

In [ ]:
POSITION_MANIFEST = ROOT / "results/manifests/analysis_position_manifest.tsv"
FIXED_SMALL_CENTROIDS = ROOT / "results/annotations/well_centroids_fixed_small_mapped.tsv"
OUT_DIR = ROOT / "results/measurements/live_fixed_small"
QC_DIR = ROOT / "results/qc/08_fixed_foxf1_dip_review"
for path in [OUT_DIR, QC_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FIXED_COHORT_ID = "2026-01-22_day2_fix"
TARGET_MEASUREMENT = "fixed_tagyfp_bgz_over_dapi_gate"
DIP_POSITIONS = ["1-2", "3-2"]
REFERENCE_POSITIONS = ["5-5"]
REVIEW_POSITIONS = DIP_POSITIONS + REFERENCE_POSITIONS
DIP_WINDOW_UM = (700.0, 800.0)
TRACE_WINDOW_PAD_UM = 120.0

FIXED_GLOBAL_BG_TSV = OUT_DIR / "fixed_global_background_params.tsv"
FIXED_DAPI_GATE_SUMMARY_TSV = OUT_DIR / "fixed_dapi_gate_summary.tsv"
FIXED_FOXF1_PIXEL_BIN_STATS_TSV = OUT_DIR / "fixed_foxf1_pixel_bin_stats.tsv"
FIXED_FOXF1_TRACE_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_foxf1_distance_trace_across_images.tsv"
FIXED_MEASUREMENT_SUMMARY_TXT = OUT_DIR / "fixed_measurement_summary.txt"

TRACE_CONTEXT_PNG = QC_DIR / "fixed_foxf1_dip_trace_context.png"
SPATIAL_OVERVIEW_PNG = QC_DIR / "fixed_foxf1_dip_spatial_overview.png"
BIN_REVIEW_PNGS = {pos: QC_DIR / f"{pos}_fixed_foxf1_dip_bin_review.png" for pos in DIP_POSITIONS}
DIP_SUMMARY_TSV = OUT_DIR / "fixed_foxf1_dip_review_summary.tsv"


## Load Shared 07 Outputs

In [ ]:
required_outputs = [
    POSITION_MANIFEST,
    FIXED_SMALL_CENTROIDS,
    FIXED_GLOBAL_BG_TSV,
    FIXED_DAPI_GATE_SUMMARY_TSV,
    FIXED_FOXF1_PIXEL_BIN_STATS_TSV,
    FIXED_FOXF1_TRACE_ACROSS_IMAGES_TSV,
    FIXED_MEASUREMENT_SUMMARY_TXT,
]
missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError("Run 07 first. Missing outputs:\n" + "\n".join(missing))

pos_df = common.load_position_manifest(POSITION_MANIFEST)
fixed_pos_df = common.filter_position_manifest(
    pos_df=pos_df,
    cohort_ids=[FIXED_COHORT_ID],
    conditions=["fixed"],
).sort_values(["canonical_position"]).reset_index(drop=True)
fixed_pos_df["canonical_position"] = fixed_pos_df["canonical_position"].astype(str)
fixed_pos_by_cp = {str(r.canonical_position): pd.Series(r._asdict()) for r in fixed_pos_df.itertuples(index=False)}

fixed_cent_df = pd.read_csv(FIXED_SMALL_CENTROIDS, sep="	")
fixed_cent_df["canonical_position"] = fixed_cent_df["canonical_position"].astype(str)

fixed_global_bg_df = pd.read_csv(FIXED_GLOBAL_BG_TSV, sep="	")
fixed_dapi_gate_df = pd.read_csv(FIXED_DAPI_GATE_SUMMARY_TSV, sep="	")
fixed_dapi_gate_df["canonical_position"] = fixed_dapi_gate_df["canonical_position"].astype(str)
foxf1_stats_df = pd.read_csv(FIXED_FOXF1_PIXEL_BIN_STATS_TSV, sep="	")
foxf1_stats_df["canonical_position"] = foxf1_stats_df["canonical_position"].astype(str)
foxf1_trace_df = pd.read_csv(FIXED_FOXF1_TRACE_ACROSS_IMAGES_TSV, sep="	")

summary_map = {}
for line in FIXED_MEASUREMENT_SUMMARY_TXT.read_text().splitlines():
    if not line.strip() or "	" not in line:
        continue
    key, value = line.split("	", 1)
    summary_map[key] = value

final04 = lfq.build_final04_fixed_dapi_masks(
    position_manifest=POSITION_MANIFEST,
    cohort_id=FIXED_COHORT_ID,
    root=ROOT,
)
fixed_mask_payloads = final04["mask_payloads"]

dapi_gate_threshold = float(fixed_global_bg_df.loc[fixed_global_bg_df["channel_key"] == "dapi", "analysis_gate_threshold"].iloc[0])
foxf1_mu = float(fixed_global_bg_df.loc[fixed_global_bg_df["channel_key"] == "tagyfp", "mu_bg_raw"].iloc[0])
foxf1_sigma = float(fixed_global_bg_df.loc[fixed_global_bg_df["channel_key"] == "tagyfp", "sigma_bg_raw"].iloc[0])

dip_bins_df = foxf1_trace_df[
    foxf1_trace_df["bin_mid_um"].between(DIP_WINDOW_UM[0], DIP_WINDOW_UM[1], inclusive="both")
].copy().sort_values("bin_mid_um").reset_index(drop=True)
if dip_bins_df.empty:
    raise RuntimeError("No fixed FOXF1 bins found in the requested dip window.")

display(dip_bins_df[["bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um", "mean", "sd", "n_images"]])
print("Measurement-level FOXF1 exclusions:", summary_map.get("fixed_tagyfp_position_exclusions", ""))
print("Review positions:", REVIEW_POSITIONS)
print("DAPI gate threshold:", dapi_gate_threshold)
print("FOXF1 background params:", foxf1_mu, foxf1_sigma)


## Rebuild FOXF1 Review Payloads

In [ ]:
def _mask_contour(ax, mask: np.ndarray, color: str = "cyan", lw: float = 0.8) -> None:
    if np.any(mask):
        ax.contour(mask.astype(np.float32), levels=[0.5], colors=[color], linewidths=lw)


def _scatter_beads(ax, xy: np.ndarray, color: str = "magenta") -> None:
    if xy.size:
        ax.scatter(xy[:, 0], xy[:, 1], s=34, facecolors="none", edgecolors=color, linewidths=1.1)


def _channel_bg_standardize(raw: np.ndarray, mu_bg: float, sigma_bg: float) -> np.ndarray:
    return ((np.asarray(raw, dtype=np.float32) - float(mu_bg)) / max(float(sigma_bg), float(lfq.EPS))).astype(np.float32)


def _beads_for_position(pos: str) -> tuple[np.ndarray, np.ndarray]:
    cent_sub = fixed_cent_df[
        (fixed_cent_df["canonical_position"].astype(str) == str(pos))
        & (fixed_cent_df["mapping_status"] == "ok")
        & (fixed_cent_df["annotation_status"] == "annotated")
    ].copy()
    all_xy = cent_sub[["centroid_fixed_small_x_px", "centroid_fixed_small_y_px"]].to_numpy(dtype=np.float32)
    disp_sub = cent_sub[cent_sub["inside_fixed_small_fov"].fillna(False).astype(bool)].copy()
    display_xy = disp_sub[["centroid_fixed_small_x_px", "centroid_fixed_small_y_px"]].to_numpy(dtype=np.float32)
    return all_xy, display_xy


def _build_review_payload(pos: str) -> dict:
    gate_row = fixed_dapi_gate_df[fixed_dapi_gate_df["canonical_position"].astype(str) == str(pos)].iloc[0]
    if float(gate_row.get("foxf1_manual_exclusion_px", 0)) != 0:
        raise RuntimeError(f"{pos} has a FOXF1 manual exclusion in 07; this review notebook is only for non-excluded positions.")
    if str(pos) in set(filter(None, map(str.strip, str(summary_map.get("fixed_tagyfp_position_exclusions", "")).split(";")))):
        raise RuntimeError(f"{pos} is excluded from downstream fixed FOXF1 measurement in 07.")

    fixed_row = fixed_pos_by_cp[str(pos)]
    img = lfq.read_czi_with_optional_fixed_small_plane_selection(
        ROOT / str(fixed_row["primary_analysis_file"]),
        canonical_position=str(pos),
    )
    dapi_idx = common.find_channel_index(img.channels, ["dapi"])
    tagyfp_idx = common.find_channel_index(img.channels, ["tagyfp", "foxf1", "yfp"])
    if dapi_idx is None or tagyfp_idx is None:
        raise RuntimeError(f"Missing DAPI or FOXF1 channel for {pos}: {img.channels}")

    dapi_raw = np.asarray(img.channel_images[int(dapi_idx)], dtype=np.float32)
    tagyfp_raw = np.asarray(img.channel_images[int(tagyfp_idx)], dtype=np.float32)
    final_mask = np.asarray(fixed_mask_payloads[str(pos)]["final_mask"], dtype=bool)

    finite_all = np.isfinite(dapi_raw) & np.isfinite(tagyfp_raw)
    dapi_gate_keep = finite_all & (dapi_raw >= dapi_gate_threshold)
    analysis_mask = np.asarray(final_mask & dapi_gate_keep, dtype=bool)

    tagyfp_bgz = _channel_bg_standardize(tagyfp_raw, foxf1_mu, foxf1_sigma)
    with np.errstate(divide="ignore", invalid="ignore"):
        tagyfp_ratio = np.asarray(tagyfp_bgz / np.maximum(dapi_raw, float(lfq.EPS)), dtype=np.float32)

    beads_xy_all, beads_xy_display = _beads_for_position(str(pos))
    if beads_xy_all.size == 0:
        raise RuntimeError(f"No mapped fixed bead centroids for {pos}")

    dist_um = common.nearest_bead_distance_map_um(
        image_shape_yx=dapi_raw.shape,
        centroid_xy_px=beads_xy_all,
        pixel_um_x=float(img.pixel_um_x),
        pixel_um_y=float(img.pixel_um_y),
    )

    residual_img = np.full(dapi_raw.shape, np.nan, dtype=np.float32)
    dip_window_mask = np.zeros(dapi_raw.shape, dtype=bool)
    bin_masks = {}
    bin_rows = []
    for rr in dip_bins_df.itertuples(index=False):
        upper_inclusive = float(rr.bin_end_um) >= float(dip_bins_df["bin_end_um"].max())
        if upper_inclusive:
            bin_mask = analysis_mask & np.isfinite(dist_um) & (dist_um >= float(rr.bin_start_um)) & (dist_um <= float(rr.bin_end_um) + 1e-9)
        else:
            bin_mask = analysis_mask & np.isfinite(dist_um) & (dist_um >= float(rr.bin_start_um)) & (dist_um < float(rr.bin_end_um))
        bin_mask = np.asarray(bin_mask, dtype=bool)
        bin_masks[float(rr.bin_mid_um)] = bin_mask
        dip_window_mask |= bin_mask
        residual_img[bin_mask] = np.asarray(tagyfp_ratio[bin_mask] - float(rr.mean), dtype=np.float32)
        vals = np.asarray(tagyfp_ratio[bin_mask], dtype=np.float32)
        neg_mask = np.asarray(bin_mask & np.isfinite(residual_img) & (residual_img < 0), dtype=bool)
        bin_rows.append(
            {
                "canonical_position": str(pos),
                "bin_idx": int(rr.bin_idx),
                "bin_start_um": float(rr.bin_start_um),
                "bin_end_um": float(rr.bin_end_um),
                "bin_mid_um": float(rr.bin_mid_um),
                "count_px": int(np.sum(bin_mask)),
                "mean_value": float(np.nanmean(vals)) if vals.size else np.nan,
                "median_value": float(np.nanmedian(vals)) if vals.size else np.nan,
                "min_value": float(np.nanmin(vals)) if vals.size else np.nan,
                "cohort_mean": float(rr.mean),
                "delta_mean": float(np.nanmean(vals) - float(rr.mean)) if vals.size else np.nan,
                "negative_px": int(np.sum(neg_mask)),
                "negative_fraction": float(np.sum(neg_mask) / max(np.sum(bin_mask), 1)),
            }
        )

    bin_summary_df = pd.DataFrame(bin_rows)
    negative_residual_mask = np.asarray(dip_window_mask & np.isfinite(residual_img) & (residual_img < 0), dtype=bool)
    return {
        "pos": str(pos),
        "dapi_raw": dapi_raw,
        "tagyfp_raw": tagyfp_raw,
        "tagyfp_bgz": tagyfp_bgz,
        "tagyfp_ratio": tagyfp_ratio,
        "final_mask": final_mask,
        "analysis_mask": analysis_mask,
        "distance_um": dist_um,
        "dip_window_mask": dip_window_mask,
        "residual_img": residual_img,
        "negative_residual_mask": negative_residual_mask,
        "bin_masks": bin_masks,
        "bin_summary_df": bin_summary_df,
        "beads_xy_all": beads_xy_all,
        "beads_xy_display": beads_xy_display,
        "pixel_um_x": float(img.pixel_um_x),
        "pixel_um_y": float(img.pixel_um_y),
    }


review_payloads = {pos: _build_review_payload(pos) for pos in REVIEW_POSITIONS}
dip_payloads = {pos: review_payloads[pos] for pos in DIP_POSITIONS}

ratio_vals = np.concatenate(
    [np.asarray(review_payloads[pos]["tagyfp_ratio"][review_payloads[pos]["analysis_mask"]], dtype=np.float32) for pos in REVIEW_POSITIONS]
)
ratio_lim = float(np.nanquantile(np.abs(ratio_vals[np.isfinite(ratio_vals)]), 0.995)) if np.any(np.isfinite(ratio_vals)) else 1.0
ratio_lim = max(ratio_lim, 1e-4)

bgz_vals = np.concatenate(
    [np.asarray(review_payloads[pos]["tagyfp_bgz"][review_payloads[pos]["analysis_mask"]], dtype=np.float32) for pos in REVIEW_POSITIONS]
)
bgz_lim = float(np.nanquantile(np.abs(bgz_vals[np.isfinite(bgz_vals)]), 0.995)) if np.any(np.isfinite(bgz_vals)) else 1.0
bgz_lim = max(bgz_lim, 1.0)

residual_vals = np.concatenate(
    [np.asarray(review_payloads[pos]["residual_img"][review_payloads[pos]["dip_window_mask"]], dtype=np.float32) for pos in DIP_POSITIONS]
)
residual_lim = float(np.nanquantile(np.abs(residual_vals[np.isfinite(residual_vals)]), 0.995)) if np.any(np.isfinite(residual_vals)) else 1e-4
residual_lim = max(residual_lim, 1e-4)

dip_review_summary_df = pd.concat([review_payloads[pos]["bin_summary_df"] for pos in DIP_POSITIONS], ignore_index=True)
dip_review_summary_df.to_csv(DIP_SUMMARY_TSV, sep="	", index=False)

print("Review payloads rebuilt:", list(review_payloads.keys()))
display(dip_review_summary_df)


## Trace Context

In [ ]:
color_map = {"1-2": "#8e0000", "3-2": "#c62828", "5-5": "#ef5350"}
fig, ax = plt.subplots(1, 1, figsize=(7.8, 5.6))
agg = foxf1_trace_df.sort_values("bin_mid_um").copy()
mean_vals = agg["mean"].to_numpy(dtype=float)
sd_vals = agg["sd"].to_numpy(dtype=float)
ax.plot(agg["bin_mid_um"], mean_vals, color="#c62828", lw=2.4, label="Cohort mean")
ax.fill_between(agg["bin_mid_um"], mean_vals - sd_vals, mean_vals + sd_vals, color="#c62828", alpha=0.18, label="Cohort ±SD")
for rr in dip_bins_df.itertuples(index=False):
    ax.axvspan(float(rr.bin_start_um), float(rr.bin_end_um), color="#fdd835", alpha=0.16, lw=0)
for pos in REVIEW_POSITIONS:
    sub = foxf1_stats_df[foxf1_stats_df["canonical_position"].astype(str) == str(pos)].copy().sort_values("bin_mid_um")
    ax.plot(sub["bin_mid_um"], sub["mean_value"], color=color_map[str(pos)], lw=2.0, alpha=0.95, label=pos)
    ax.scatter(sub["bin_mid_um"], sub["mean_value"], color=color_map[str(pos)], s=12, alpha=0.9)
ax.set_xlim(float(DIP_WINDOW_UM[0] - TRACE_WINDOW_PAD_UM), float(DIP_WINDOW_UM[1] + TRACE_WINDOW_PAD_UM))
ax.set_xlabel("Distance from nearest bead (um)")
ax.set_ylabel("Mean FOXF1-YFP reporter (fixed) / raw DAPI")
ax.set_title("Fixed FOXF1-YFP reporter / DAPI trace context around the ~750 um dip")
ax.legend(loc="best")
plt.tight_layout()
fig.savefig(TRACE_CONTEXT_PNG, dpi=180, bbox_inches="tight")
plt.show()
display(Image(filename=str(TRACE_CONTEXT_PNG)))


## Dip Window Spatial Overview

In [ ]:
fig, axes = plt.subplots(len(DIP_POSITIONS), 5, figsize=(22.0, 4.6 * len(DIP_POSITIONS)), constrained_layout=True)
if len(DIP_POSITIONS) == 1:
    axes = np.asarray([axes])

for row_idx, pos in enumerate(DIP_POSITIONS):
    payload = dip_payloads[str(pos)]
    dapi_show = np.asarray(payload["dapi_raw"], dtype=np.float32)
    bgz_show = np.where(payload["analysis_mask"], payload["tagyfp_bgz"], np.nan)
    ratio_show = np.where(payload["analysis_mask"], payload["tagyfp_ratio"], np.nan)
    residual_show = np.where(payload["dip_window_mask"], payload["residual_img"], np.nan)
    neg_overlay = np.asarray(payload["negative_residual_mask"], dtype=bool)
    window_delta = float(payload["bin_summary_df"]["delta_mean"].mean())

    ax = axes[row_idx, 0]
    ax.imshow(dapi_show, cmap="gray")
    _mask_contour(ax, payload["analysis_mask"], color="white", lw=0.7)
    _mask_contour(ax, payload["dip_window_mask"], color="#fdd835", lw=1.0)
    _scatter_beads(ax, payload["beads_xy_display"], color="magenta")
    ax.set_title(f"{pos} | raw DAPI + 700-800 um window")
    ax.axis("off")

    ax = axes[row_idx, 1]
    ax.imshow(bgz_show, cmap="RdBu_r", vmin=-bgz_lim, vmax=bgz_lim)
    _mask_contour(ax, payload["analysis_mask"], color="white", lw=0.7)
    _mask_contour(ax, payload["dip_window_mask"], color="#fdd835", lw=1.0)
    _scatter_beads(ax, payload["beads_xy_display"], color="magenta")
    ax.set_title(f"{pos} | FOXF1 bgz")
    ax.axis("off")

    ax = axes[row_idx, 2]
    ax.imshow(ratio_show, cmap="RdBu_r", vmin=-ratio_lim, vmax=ratio_lim)
    _mask_contour(ax, payload["analysis_mask"], color="white", lw=0.7)
    _mask_contour(ax, payload["dip_window_mask"], color="#fdd835", lw=1.0)
    _scatter_beads(ax, payload["beads_xy_display"], color="magenta")
    ax.set_title(f"{pos} | FOXF1 bgz / DAPI")
    ax.axis("off")

    ax = axes[row_idx, 3]
    ax.imshow(residual_show, cmap="RdBu_r", vmin=-residual_lim, vmax=residual_lim)
    _mask_contour(ax, payload["analysis_mask"], color="white", lw=0.6)
    _mask_contour(ax, payload["dip_window_mask"], color="#fdd835", lw=1.0)
    _scatter_beads(ax, payload["beads_xy_display"], color="magenta")
    ax.set_title(f"{pos} | residual to cohort bin mean")
    ax.axis("off")

    ax = axes[row_idx, 4]
    ax.imshow(ratio_show, cmap="RdBu_r", vmin=-ratio_lim, vmax=ratio_lim)
    _mask_contour(ax, payload["analysis_mask"], color="white", lw=0.6)
    _mask_contour(ax, payload["dip_window_mask"], color="#fdd835", lw=1.0)
    overlay = np.zeros((*neg_overlay.shape, 4), dtype=np.float32)
    overlay[..., 0] = 1.0
    overlay[..., 2] = 1.0
    overlay[..., 3] = neg_overlay.astype(np.float32) * 0.78
    ax.imshow(overlay)
    _scatter_beads(ax, payload["beads_xy_display"], color="magenta")
    ax.set_title(f"{pos} | negative residual pixels | mean delta={window_delta:.3g}")
    ax.axis("off")

fig.savefig(SPATIAL_OVERVIEW_PNG, dpi=180, bbox_inches="tight")
plt.show()
display(Image(filename=str(SPATIAL_OVERVIEW_PNG)))


## Bin-Resolved Dip Review

In [ ]:
for pos in DIP_POSITIONS:
    payload = dip_payloads[str(pos)]
    fig, axes = plt.subplots(2, len(dip_bins_df), figsize=(4.5 * len(dip_bins_df), 8.0), constrained_layout=True)
    if len(dip_bins_df) == 1:
        axes = np.asarray(axes).reshape(2, 1)
    ratio_show = np.where(payload["analysis_mask"], payload["tagyfp_ratio"], np.nan)
    for col_idx, rr in enumerate(dip_bins_df.itertuples(index=False)):
        mid = float(rr.bin_mid_um)
        bin_mask = np.asarray(payload["bin_masks"][mid], dtype=bool)
        residual_bin = np.where(bin_mask, payload["residual_img"], np.nan)
        summary_row = payload["bin_summary_df"].loc[payload["bin_summary_df"]["bin_mid_um"] == mid].iloc[0]

        ax = axes[0, col_idx]
        ax.imshow(ratio_show, cmap="RdBu_r", vmin=-ratio_lim, vmax=ratio_lim)
        _mask_contour(ax, payload["analysis_mask"], color="white", lw=0.6)
        _mask_contour(ax, bin_mask, color="#fdd835", lw=1.1)
        neg_bin = np.asarray(bin_mask & np.isfinite(payload["residual_img"]) & (payload["residual_img"] < 0), dtype=bool)
        overlay = np.zeros((*neg_bin.shape, 4), dtype=np.float32)
        overlay[..., 0] = 1.0
        overlay[..., 2] = 1.0
        overlay[..., 3] = neg_bin.astype(np.float32) * 0.75
        ax.imshow(overlay)
        _scatter_beads(ax, payload["beads_xy_display"], color="magenta")
        ax.set_title(f"{pos} | {mid:.1f} um\nmean={summary_row['mean_value']:.3g} vs cohort={summary_row['cohort_mean']:.3g}")
        ax.axis("off")

        ax = axes[1, col_idx]
        ax.imshow(residual_bin, cmap="RdBu_r", vmin=-residual_lim, vmax=residual_lim)
        _mask_contour(ax, bin_mask, color="#fdd835", lw=1.1)
        _scatter_beads(ax, payload["beads_xy_display"], color="magenta")
        ax.set_title(f"delta={summary_row['delta_mean']:.3g} | neg frac={summary_row['negative_fraction']:.2%}")
        ax.axis("off")

    out_png = BIN_REVIEW_PNGS[str(pos)]
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.show()
    display(Image(filename=str(out_png)))


## Dip Window Summary

In [ ]:
display(dip_review_summary_df)
print("Saved:", DIP_SUMMARY_TSV)
